# Web Scraping Pipeline for NASA Blog Articles

A scraping pipeline that collects articles from NASA's science blog archive, pulling out the metadata for each post (title, summary, author, date, category, images) and structuring everything into a single Pandas dataframe.

This run collects articles from the first page of the blog archive (10 posts). The scraper for each individual article page is written to work on any post URL, so it's a small change to add pagination and pull more pages — that's noted at the end as a natural next step, not something this version does yet.


## 1. Setup

In [ ]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

OUTPUT_DIR = "./output"


## 2. Collecting Article Links from the Blog Archive

The archive page lists posts as `<article>` elements. For each one we grab the link, title, summary, and thumbnail image using CSS selectors matched to the page's structure.


In [ ]:
archive_url = "https://science.nasa.gov/blogs/"
site = requests.get(archive_url)
soup = BeautifulSoup(site.text, 'html.parser')

link_tags = soup.select("#blog-archive-content-feed > ul > article > a.button-primary.link-external-false")
link = [tag['href'] for tag in link_tags]

title_tags = soup.select("#blog-archive-content-feed > ul > article > h2")
title = [tag.text for tag in title_tags]

summary_tags = soup.select("#blog-archive-content-feed > ul > article > div > p")
summary = [tag.text for tag in summary_tags]

img_tags = soup.select("#blog-archive-content-feed > ul > article > a:nth-child(3) > img")
img_src = [tag['src'] for tag in img_tags]

print(f"Found {len(link)} articles on the archive page")


## 3. Scraping Each Article Page

Each article page has more detail than the archive listing: an image caption, the full body text (as a list of paragraphs), the author's name and avatar, the publish date, and category tags. These selectors are matched to NASA's individual blog post layout, which is a different structure from the archive page above.


In [ ]:
caption, description, auther_name, auther_img, date, category = [], [], [], [], [], []

article_selector_base = (
    r"body > div.grid-container.grid-container-extrawide.padding-x-4.padding-y-6."
    r"desktop\:padding-y-8 > article > div.desktop\:grid-col-8"
)

for article_link in tqdm(link, desc="Scraping articles"):
    site = requests.get(article_link)
    soup = BeautifulSoup(site.text, "html.parser")

    temp = soup.select(
        f"{article_selector_base} > div > div.hds-media.hds-module.wp-block-image "
        "> div > div > figcaption > div.hds-caption-text.p-sm.margin-0"
    )
    caption.append([p.get_text() for p in temp])

    temp = soup.select(f"{article_selector_base} > div > p")
    description.append([p.get_text() for p in temp])

    temp = soup.select(f"{article_selector_base} > div > div > div > div:nth-child(2) > p")
    auther_name.append(temp[0].text if temp else None)

    temp = soup.select(f"{article_selector_base} > div > div > div > img")
    auther_img.append(temp[0]['src'] if temp else None)

    temp = soup.select(f"{article_selector_base} > div > div > div:nth-child(2) > time")
    date.append([t.text for t in temp])

    temp = soup.select(f"{article_selector_base} > div > div > div:nth-child(3) > nav > a")
    category.append([c.text for c in temp])


## 4. Building the Final Dataframe

In [ ]:
data = pd.DataFrame({
    'title': title,
    'summary': summary,
    'img_src': img_src,
    'caption': caption,
    'description': description,
    'auther_name': auther_name,
    'auther_img': auther_img,
    'date': date,
    'category': category,
    'link': link
})

data


## 5. Exporting to CSV

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

data.to_csv(f"{OUTPUT_DIR}/web_crawling_nasa.csv", index=None, encoding='utf-8')
print(f"Saved {len(data)} articles to {OUTPUT_DIR}/web_crawling_nasa.csv")


## 6. Downloading Images

Both the author avatars and the article thumbnail images are downloaded locally. Requests are wrapped in a retry so a single failed download doesn't stop the whole run.


In [ ]:
def download_image(url, path, retries=1):
    for attempt in range(retries + 1):
        try:
            response = requests.get(url)
            with open(path, 'wb') as f:
                f.write(response.content)
            return
        except requests.RequestException:
            if attempt < retries:
                time.sleep(5)
            else:
                print(f"Failed to download {url}")


os.makedirs(f"{OUTPUT_DIR}/author_images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/blog_images", exist_ok=True)

authors = list(zip(auther_name, auther_img))
for name, img_url in tqdm(authors, desc="Downloading author images"):
    if img_url:
        download_image(img_url, f"{OUTPUT_DIR}/author_images/{name}.png")

for i, img_url in enumerate(tqdm(img_src, desc="Downloading blog images")):
    download_image(img_url, f"{OUTPUT_DIR}/blog_images/{i}.png")


## Notes & Next Steps

- This run pulls the 10 articles listed on the archive page's first load. The archive page loads more posts as you scroll (infinite scroll / pagination), so the natural next step is to detect and follow that mechanism to collect more articles.
- Selectors are matched to NASA's current page structure and will need updating if the site's HTML changes.
- Author name/image extraction assumes a single author per post; posts with multiple authors would need a small adjustment to the selector logic.
